# Week 02 — Data Analytics and the Need

**Spatial Data Science with Python · MSc GIS / AGMS 1333**  
**Official Table 4 topic:** Introduction to Data Science · Descriptive, diagnostic, predictive, prescriptive analytics

**Learning outcome.** Compare facility provision and nearest-facility distance.

**What you will implement:** Grouping and aggregation, rates, Euclidean distance.  
**Mathematical model:** Rate per 10,000 = 10,000 × facilities / population.

> **Data provenance:** The Urban Teaching Grid is a **synthetic** 12 × 12 grid at 250-m cell spacing. Its temperatures and facilities are simulated; do not present its numerical outputs as field measurements. All distance calculations in the labs use the projected coordinates EPSG:32647 where relevant.

**How to run.** In Colab, upload this notebook from *File → Upload notebook*. Run each cell top-to-bottom. The setup cell below will ask for `DATA_ONLY.zip` only if the data folder is absent. In JupyterLab, extract the complete ZIP and open this notebook in `01_Core_Labs/`.


## 1. Prepare the environment and locate the data

The same setup works in JupyterLab and Google Colab. It selects an existing local data directory, or prompts for the packaged dataset in Colab. `assert` fails early if the archive is incomplete.

In [ ]:
from pathlib import Path
import io
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Local Jupyter: data/ or ../data. Colab: upload DATA_ONLY.zip once per runtime.
candidates = [Path('data'), Path('../data'), Path('/content/data')]
DATA = next((p for p in candidates if (p / 'cells.csv').is_file()), None)
if DATA is None:
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError('Cannot find data/. Extract the course ZIP and keep the data folder with the notebooks.') from exc
    print('Select DATA_ONLY.zip from the downloaded course package:')
    uploaded = files.upload()
    zip_name = next((name for name in uploaded if name.endswith('.zip')), None)
    if zip_name is None:
        raise FileNotFoundError('Please upload DATA_ONLY.zip and run this cell again.')
    with zipfile.ZipFile(io.BytesIO(uploaded[zip_name])) as archive:
        archive.extractall(Path.cwd())
    DATA = Path('data')
assert (DATA / 'metadata.json').is_file(), 'Data package incomplete.'
print('Dataset folder:', DATA.resolve())
print('Synthetic teaching data: not field observations.')


## 2. Key concepts and expected data

The analysis follows **input → cleaning → computation → validation → interpretation**. Always inspect observation keys, coordinate units, missing values, and the intended study period before fitting a model.

**Mathematical foundation:** Rate per 10,000 = 10,000 × facilities / population

**Method family:** Grouping and aggregation, rates, Euclidean distance.

## 3. Main computation

Run this cell after the setup. The block calculates the principal statistics or fits the weekly model. The original course code is retained so results remain aligned with the slide examples.

In [ ]:
cells = pd.read_csv(DATA/'cells.csv'); fac = pd.read_csv(DATA/'facilities.csv')
fac['col'] = ((fac.easting_m - 797000)//250).astype(int)
fac['row'] = ((fac.northing_m - 345000)//250).astype(int)
counts = fac.groupby(['row','col']).size().rename('facility_count').reset_index()
a = cells.merge(counts,on=['row','col'],how='left'); a['facility_count']=a.facility_count.fillna(0)
rates = a.groupby('sector')[['facility_count','population']].sum()
rates['per_10000'] = 10000*rates.facility_count/rates.population
print(rates.round(2));assert int(rates.facility_count.sum())==24

### Reading the main code

**Facility assignment.** Integer division `//250` maps projected coordinates to grid indices. The constants are dataset-specific; a real polygon boundary join needs a stated predicate.

- `pd.read_csv` — Reads CSV text into a pandas DataFrame. Inspect head() and column types after loading.
- `groupby` — Groups rows before counting or aggregating by sector, date, or other key.
- `merge` — Joins tables on a declared key; validate cardinality to avoid duplicate matches.
- `assert` — Checks a claimed condition and stops if it is false.

**Verification questions:** Which variables contain the inputs? What new object does the cell create? What numerical result should be checked before continuing?

## 4. Map, plot, or evaluate the result

The final block turns results into a labelled graphic or output artifact. Keep units in axes/legends, and distinguish model predictions from observations.

In [ ]:
a_xy=a[['easting_m','northing_m']].to_numpy(); f_xy=fac[['easting_m','northing_m']].to_numpy()
a['nearest_facility_m']=np.linalg.norm(a_xy[:,None,:]-f_xy[None,:,:],axis=2).min(axis=1)
plt.figure(figsize=(7,5));plt.scatter(a.easting_m,a.northing_m,c=a.nearest_facility_m,s=95,cmap='viridis')
plt.scatter(fac.easting_m,fac.northing_m,marker='x',color='red',label='Facility')
plt.colorbar(label='Straight-line distance (m)');plt.axis('equal');plt.legend();plt.show()
print('Median distance (m):',round(a.nearest_facility_m.median(),1))

### Reading the output code

**Rates.** `groupby('sector').sum()` must be applied to numerator and denominator *before* division.

- `np.linalg.norm` — Calculates Euclidean vector lengths, here interpreted as planar distances in metres.
- `plt.` — Creates a figure; title, axes, and legend are part of the analytical interpretation.

**Distance.** Broadcasting forms all cell-center/facility pairs, `np.linalg.norm` finds Euclidean distances, and `.min(axis=1)` selects the nearest point. It is not network travel time.

## 5. Critical interpretation

### Interpret your results

Complete the following short report **after** running the notebook:

1. **Research question:** What geographic question does this notebook address?
2. **Observation unit and period:** What does one row/pixel represent? What is its date and CRS?
3. **Method:** Why is this method suitable, and what assumptions does it make?
4. **Results:** Report two numerical outputs or figure patterns with units.
5. **Validity:** State one data limitation and one model/evaluation limitation.
6. **Extension:** Modify one parameter, compare results, and describe what changed.


## 6. Questions and independent practice

1. Compare sector counts, rates, and straight-line access; explain why they answer different questions.
2. What would happen if the spatial unit, study date, or CRS changed?
3. Is this analysis descriptive, inferential, predictive, or a decision-support example? Justify your answer.
4. Which assumption should be tested before use in a real project?

**Next step:** Open the matching advanced notebook in `02_Advanced_Experiments/` for four short method variations.